
## 初始化库，加载分层后的训练集😆

In [51]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

plt.style.use("default")  # 先加载默认样式
# 设置中文显示+图表样式
plt.rcParams["font.family"] = "Microsoft YaHei"
plt.rcParams["axes.unicode_minus"] = False

print("✅ 依赖库导入完成！")
# 训练集路径（需与之前「分层抽样」Notebook的保存路径一致）
train_path = "../data/processed/train_set.csv"

# 检查文件是否存在，避免路径错误
if not os.path.exists(train_path):
    print(f"❌ 错误：训练集文件未找到！请先运行「数据清洗+分层抽样」Notebook，生成 {train_path}")
else:
    # 加载训练集，保留原始索引
    df_train = pd.read_csv(train_path, index_col=0)
    print("="*60)
    print("✅ 训练集加载成功！")
    print(f"训练集规模：{df_train.shape[0]} 行 × {df_train.shape[1]} 列")
    print(f"目标变量（违约标识）：SeriousDlqin2yrs（1=违约，0=正常）")
    print("="*60)

✅ 依赖库导入完成！
✅ 训练集加载成功！
训练集规模：119780 行 × 11 列
目标变量（违约标识）：SeriousDlqin2yrs（1=违约，0=正常）


## 训练集缺失值分析（明确填充对象）

根据EDA可知，月收入和家属收入有数据缺失
- 针对MonthlyIncome（月收入）字段中存在的 0 值
    - 业务层面：信用卡授信要求用户具备稳定还款能力，月收入为 0 的样本不具备真实业务代表性；
    - 数据层面：0 值会使收入分布极端偏斜，会干扰模型对还款能力特征的学习；
    - 处理方案：采用中位数替换策略
- 针对 NumberOfDependents（家属数量）
    - 出现一个断层的20家属，13个家属：同样极不合理，并且都只有1个，建议处理。
    - 10个家属：理论上可能存在。考虑到只有5个样本，可以保留，但需警惕其对模型的潜在影响。
    - NA值：缺失值占比约（3924/总样本数），需要妥善处理。

- 针对 NumberRealEstateLoansOrLines（房产数量）
    - 出现54的过大值，可能删除

In [52]:
# 统计训练集缺失值（仅展示有缺失的字段）
missing_stats = pd.DataFrame({
    "缺失数量": df_train.isnull().sum(),
    "缺失比例(%)": (df_train.isnull().sum() / len(df_train) * 100).round(2)
}).query("缺失数量 > 0")  # 过滤无缺失字段

print("📊 训练集缺失值统计：")
print(missing_stats)

# 若无缺失值，直接跳过后续填充步骤
if missing_stats.empty:
    df_train_filled = df_train.copy()
    print("\n✅ 训练集无缺失值，无需填充！")
else:
    print("\n⚠️  需填充字段：")
    for col in missing_stats.index:
        print(f"- {col}：缺失 {missing_stats.loc[col, '缺失数量']} 条（{missing_stats.loc[col, '缺失比例(%)']}%）")

# 复制训练集，避免修改原始数据
df_train_filled = df_train.copy()

# 1. 定义填充规则（仅基于训练集计算参数，无数据泄露）
fill_rules = {
    "MonthlyIncome": df_train["MonthlyIncome"].median(),  # 月收入：训练集中位数（抗异常值）
    "NumberOfDependents": 0  # 家属人数：按需求固定填0
}

print("🔧 训练集缺失值填充规则：")
for col, val in fill_rules.items():
    if col == "MonthlyIncome":
        print(f"- {col}（月收入）：用训练集中位数 {val:.2f} 填充")
    else:
        print(f"- {col}（家属人数）：按需求固定填 {val}")

# 2. 执行填充
fill_log = []  # 记录填充结果
for col, val in fill_rules.items():
    fill_count = df_train[col].isnull().sum()
    df_train_filled[col] = df_train_filled[col].fillna(val)
    fill_log.append(f"- {col}：填充 {fill_count} 条缺失值")

# 3. 验证填充结果
after_fill_missing = df_train_filled.isnull().sum().sum()
print("\n" + "="*40)
print("✅ 缺失值填充完成！")
print("\n".join(fill_log))
print(f"\n📌 填充后验证：训练集总缺失值 = {after_fill_missing} 条（预期为0）")
print("="*40)

📊 训练集缺失值统计：
                     缺失数量  缺失比例(%)
MonthlyIncome       23811    19.88
NumberOfDependents   3127     2.61

⚠️  需填充字段：
- MonthlyIncome：缺失 23811 条（19.88%）
- NumberOfDependents：缺失 3127 条（2.61%）
🔧 训练集缺失值填充规则：
- MonthlyIncome（月收入）：用训练集中位数 5400.00 填充
- NumberOfDependents（家属人数）：按需求固定填 0

✅ 缺失值填充完成！
- MonthlyIncome：填充 23811 条缺失值
- NumberOfDependents：填充 3127 条缺失值

📌 填充后验证：训练集总缺失值 = 0 条（预期为0）


## 训练集硬截尾处理（DebtRatio + 循环额度使用率）
针对 DebtRatio存在过多异常大的值（异常20.87%）
 - 如此巨大的数值极不合理，可能由于收入接近0或数据录入错误导致
 - 使用**截尾**（winsorize / clip）：保留样本，但把极端值压到一个合理范围。
    - 小于下限的值 → 强制改成下限，
    - 大于上限的值 → 强制改成上限
- 使用**log 转换**（log transform）对变量做数学变换，让分布更平滑。

针对 RevolvingUtilizationOfUnsecuredLines（循环额度使用率）
- 通常应在0~1之间（超过1表示超额使用，但一般不会太大）
- IQR正常范围为[-0.76, 1.35]，实际最大值50,708，异常比例0.51%。显然，50,708是极端错误值，可能由于分母过小或数据录入错误

虽然部分极端值可能反映真实风险，但为了模型稳定性与可解释性，采用截尾处理
- DebtRatio使用硬截尾（0-10），负债是收入的10倍，再多就没有使用的意义了
- 循环额度使用率使用硬截尾（0-25）



In [53]:
# 复制填充后的数据集，用于截尾操作（后续单独保存，覆盖原列无风险）
df_train_processed = df_train_filled.copy()

print("🔧 开始执行训练集硬截尾处理（直接覆盖原列，便于EDA复用）：")
print("="*60)

# --------------------------
# 1. DebtRatio（债务率）截尾 [0, 10] → 直接覆盖原列
# --------------------------
debt_col = "DebtRatio"
# 先保存原始数据用于统计（覆盖后就拿不到原始值了）
debt_original = df_train_processed[debt_col].copy()
# 执行截尾并覆盖原列
df_train_processed[debt_col] = df_train_processed[debt_col].clip(lower=0, upper=10)

# 统计债务率截尾效果（用保存的原始数据计算）
debt_clip_below = (debt_original < 0).sum()  # <0的异常样本
debt_clip_above = (debt_original > 10).sum()  # >10的噪声样本
debt_total_clip = debt_clip_below + debt_clip_above

print(f"1. {debt_col}（债务率）截尾结果（覆盖原列）：")
print(f"   - 截尾前范围：[{debt_original.min():.2f}, {debt_original.max():.2f}]")
print(f"   - 截尾后范围：[{df_train_processed[debt_col].min():.2f}, {df_train_processed[debt_col].max():.2f}]")
print(f"   - 截尾样本：{debt_total_clip} 条（<0：{debt_clip_below} 条，>10：{debt_clip_above} 条）")
print(f"   - 截尾占比：{debt_total_clip / len(df_train_processed):.2%}")

# --------------------------
# 2. 循环额度使用率截尾 [0, 1] → 直接覆盖原列
# --------------------------
util_col = "RevolvingUtilizationOfUnsecuredLines"
# 先保存原始数据用于统计
util_original = df_train_processed[util_col].copy()
# 执行截尾并覆盖原列
df_train_processed[util_col] = df_train_processed[util_col].clip(lower=0, upper=1)

# 统计额度使用率截尾效果（用保存的原始数据计算）
util_clip_below = (util_original < 0).sum()  # <0的异常样本
util_clip_above = (util_original > 1).sum()  # >1的错误样本
util_total_clip = util_clip_below + util_clip_above

print(f"\n2. {util_col}（循环额度使用率）截尾结果（覆盖原列）：")
print(f"   - 截尾前范围：[{util_original.min():.2f}, {util_original.max():.2f}]")
print(f"   - 截尾后范围：[{df_train_processed[util_col].min():.2f}, {df_train_processed[util_col].max():.2f}]")
print(f"   - 截尾样本：{util_total_clip} 条（<0：{util_clip_below} 条，>1：{util_clip_above} 条）")
print(f"   - 截尾占比：{util_total_clip / len(df_train_processed):.2%}")

print("\n" + "="*60)
print("✅ 训练集硬截尾处理完成！")
print(f"处理方式：直接覆盖原列（{debt_col}、{util_col}），可直接复用原有EDA分析逻辑")
print(f"后续操作：保存 df_train_processed 即可，原数据未污染（基于副本处理）")
print("="*60)

🔧 开始执行训练集硬截尾处理（直接覆盖原列，便于EDA复用）：
1. DebtRatio（债务率）截尾结果（覆盖原列）：
   - 截尾前范围：[0.00, 329664.00]
   - 截尾后范围：[0.00, 10.00]
   - 截尾样本：23205 条（<0：0 条，>10：23205 条）
   - 截尾占比：19.37%

2. RevolvingUtilizationOfUnsecuredLines（循环额度使用率）截尾结果（覆盖原列）：
   - 截尾前范围：[0.00, 29110.00]
   - 截尾后范围：[0.00, 1.00]
   - 截尾样本：2665 条（<0：0 条，>1：2665 条）
   - 截尾占比：2.22%

✅ 训练集硬截尾处理完成！
处理方式：直接覆盖原列（DebtRatio、RevolvingUtilizationOfUnsecuredLines），可直接复用原有EDA分析逻辑
后续操作：保存 df_train_processed 即可，原数据未污染（基于副本处理）


## 固定间隔折线图（age + DebtRatio + 月收入 + 循环额度使用率）

In [54]:
# 设置图表大小（适合毕设报告，宽16，高12）
plt.figure(figsize=(16, 12))

# 定义4个目标字段的「固定间隔」和配置（贴合业务逻辑）
field_config = {
    # 字段名：(间隔步长, 标题, 颜色, y轴标签)
    "age": (5, "年龄分布（固定间隔：5岁）", "#2ecc71", "样本数量"),
    "DebtRatio": (1, "债务率分布（固定间隔：1）", "#e67e22", "样本数量"),
    "MonthlyIncome": (1000, "月收入分布（固定间隔：1000，最高30000）", "#9b59b6", "样本数量"),  # 步长改为1000
    "RevolvingUtilizationOfUnsecuredLines": (0.1, "循环额度使用率分布（固定间隔：0.1）", "#3498db", "样本数量")
}

# 循环绘制4个子图（2行2列）
for i, (field, (step, title, color, y_label)) in enumerate(field_config.items(), 1):
    if field == "MonthlyIncome":
        # ---------- 月收入特殊处理：1000元/档，最高30000，超30000归为一类 ----------
        min_val = 0
        max_val = 30000
        step = 1000
        # 生成bins：0,1000,...,30000，最后加一个无穷大区间容纳>30000的样本
        bins = list(np.arange(min_val, max_val + step, step))  # [0, 1000, ..., 30000]
        bins.append(np.inf)  # 最后一个区间：[30000, +∞)

        # 用pd.cut分箱，统计每个区间样本数（right=False表示左闭右开）
        interval_counts = pd.cut(
            df_train_processed[field],
            bins=bins,
            right=False
        ).value_counts(sort=False).reset_index(name="count")

        # 🔥 关键修复：动态获取分箱列的列名（不再硬编码"index"）
        interval_col = interval_counts.columns[0]  # 第一列就是存储Interval的列

        # 计算区间中点：最后一个区间（≥30000）手动设为35000，方便绘图
        interval_mid = []
        for interval in interval_counts[interval_col]:  # 用动态列名访问
            if interval.right == np.inf:
                interval_mid.append(30000 + 5000)  # 让≥30000的点在30000右侧
            else:
                interval_mid.append((interval.left + interval.right) / 2)
    else:
        # ---------- 其他字段保持原有逻辑 ----------
        min_val = df_train_processed[field].min()
        max_val = df_train_processed[field].max()
        bins = np.arange(min_val, max_val + step, step)
        interval_counts = df_train_processed[field].value_counts(
            bins=bins,
            sort=False
        ).reset_index(name="count")
        interval_col = interval_counts.columns[0]
        interval_mid = interval_counts[interval_col].map(lambda x: (x.left + x.right) / 2)

    # 绘制折线图
    plt.subplot(2, 2, i)
    plt.plot(interval_mid, interval_counts["count"], color=color, marker="o", linewidth=2, markersize=4)

    # 图表美化
    plt.title(title, fontsize=12, fontweight="bold")
    plt.xlabel(field, fontsize=10)
    plt.ylabel(y_label, fontsize=10)
    plt.grid(alpha=0.3)

    # 月收入x轴标签特殊处理：最后一个显示为「≥30000」
    if field == "MonthlyIncome":
        # 用动态列名遍历，不再写死"index"
        xticklabels = [f"{int(interval.left)}" for interval in interval_counts[interval_col][:-1]] + ["≥30000"]
        plt.xticks(interval_mid, xticklabels, rotation=45)
    else:
        plt.xticks(rotation=45 if field == "MonthlyIncome" else 0)

# 调整子图间距，避免重叠
plt.tight_layout()

# 保存图表（高分辨率300dpi，适合毕设插入）
img_path = "../data/output-img/训练集核心字段固定间隔分布.png"
plt.savefig(img_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"✅ 固定间隔折线图已保存！路径：{img_path}")
print("\n📌 图表说明：")
print("- 年龄：每5岁一个间隔，展示年龄分布集中度")
print("- 债务率：每1个单位一个间隔，展示截尾后分布")
print("- 月收入：每1000元一个间隔，0-30000元细分，≥30000元单独归类")
print("- 循环额度使用率：每0.1一个间隔，展示使用率分布峰值")

✅ 固定间隔折线图已保存！路径：../data/output-img/训练集核心字段固定间隔分布.png

📌 图表说明：
- 年龄：每5岁一个间隔，展示年龄分布集中度
- 债务率：每1个单位一个间隔，展示截尾后分布
- 月收入：每1000元一个间隔，0-30000元细分，≥30000元单独归类
- 循环额度使用率：每0.1一个间隔，展示使用率分布峰值


## 保存处理后的训练集（供后续模型训练）

In [55]:
# 保存最终处理后的训练集（缺失值填充+硬截尾）
processed_train_path = "../data/processed/train_set_processed.csv"
df_train_processed.to_csv(processed_train_path, index=True)

# 保存处理参数（供测试集复用，避免数据泄露）
process_params = {
    "missing_fill": {
        "MonthlyIncome_median": fill_rules["MonthlyIncome"],
        "NumberOfDependents_fill": fill_rules["NumberOfDependents"]
    },
    "clip_rules": {
        "DebtRatio_clip_range": [0, 10],
        "RevolvingUtil_clip_range": [0, 1]
    },
    "process_date": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
}

# 用DataFrame格式保存参数（便于后续读取）
params_df = pd.DataFrame([process_params])
params_path = "../data/processed/train_process_params.csv"
params_df.to_csv(params_path, index=False)

print("="*60)
print("✅ 训练集全流程处理完成！")
print(f"1. 处理后训练集：{processed_train_path}")
print(f"2. 处理参数记录：{params_path}（测试集需复用此参数！）")
print(f"3. 分布图表：../data/output-img/训练集核心字段固定间隔分布2.png")
print("="*60)

✅ 训练集全流程处理完成！
1. 处理后训练集：../data/processed/train_set_processed.csv
2. 处理参数记录：../data/processed/train_process_params.csv（测试集需复用此参数！）
3. 分布图表：../data/output-img/训练集核心字段固定间隔分布2.png
